# 📊 Exploratory Data Analysis
## Disease Prediction System

This notebook performs comprehensive EDA on all three datasets:
1. Heart Disease (UCI)
2. Diabetes (Pima Indians)
3. Breast Cancer Wisconsin (Diagnostic)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import load_dataset, load_all_datasets
from src.preprocessing import detect_missing_values, detect_outliers_iqr
from src.config import FIGURES_DIR

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete!')

## 1. Load All Datasets

In [ ]:
# Load datasets
datasets = load_all_datasets()

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name.upper()}")
    print(f"Shape: {df.shape}")
    print(f"Target distribution:\n{df['target'].value_counts()}")
    print(f"{'='*60}")

## 2. Dataset Overview

In [ ]:
def dataset_overview(df, name):
    """Generate comprehensive dataset overview."""
    print(f"\n{'='*60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape}")
    print(f"\nData Types:")
    print(df.dtypes.value_counts())
    print(f"\nStatistical Summary:")
    display(df.describe().round(3))
    print(f"\nFirst 5 Rows:")
    display(df.head())
    return df

for name, df in datasets.items():
    dataset_overview(df, name)

## 3. Missing Value Analysis

In [ ]:
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    missing_report = detect_missing_values(df)
    if missing_report.empty:
        print("No missing values found!")
    else:
        display(missing_report)
    
    # Check zeros in diabetes dataset
    if name == 'diabetes':
        print("\nZero values (potentially missing):")
        zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
        existing_cols = [c for c in zero_cols if c in df.columns]
        if existing_cols:
            print((df[existing_cols] == 0).sum())

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

labels_map = {
    'heart': ['No Disease', 'Disease'],
    'diabetes': ['Non-Diabetic', 'Diabetic'],
    'cancer': ['Benign', 'Malignant']
}

for idx, (name, df) in enumerate(datasets.items()):
    counts = df['target'].value_counts().sort_index()
    colors = ['#2ecc71', '#e74c3c']
    axes[idx].pie(
        counts, labels=labels_map[name], autopct='%1.1f%%',
        colors=colors, startangle=90, explode=(0.05, 0.05)
    )
    axes[idx].set_title(f'{name.upper()} - Class Distribution', fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved!')

## 5. Correlation Heatmaps

In [ ]:
for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=[np.number])
    
    fig, ax = plt.subplots(figsize=(12, 10))
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    sns.heatmap(
        corr, mask=mask, annot=True if corr.shape[0] <= 15 else False,
        cmap='RdBu_r', center=0, square=True,
        linewidths=0.5, fmt='.2f', ax=ax
    )
    ax.set_title(f'Correlation Heatmap - {name.upper()}', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'correlation_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: correlation_{name}.png')

## 6. Feature Distributions

In [ ]:
for name, df in datasets.items():
    features = [c for c in df.columns if c != 'target']
    n_features = len(features)
    n_cols = 4
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
    axes = axes.flatten()
    
    for idx, col in enumerate(features):
        if df[col].dtype in ['object']:
            continue
        sns.histplot(data=df, x=col, hue='target', ax=axes[idx], kde=True, alpha=0.6)
        axes[idx].set_title(col, fontsize=10)
        axes[idx].legend(['0', '1'])
    
    # Hide unused axes
    for idx in range(n_features, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'Feature Distributions - {name.upper()}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'distributions_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: distributions_{name}.png')

## 7. Boxplots (Outlier Visualization)

In [ ]:
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    features = [c for c in numeric_cols if c != 'target']
    
    n_features = len(features)
    n_cols = 4
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
    axes = axes.flatten()
    
    for idx, col in enumerate(features):
        sns.boxplot(data=df, x='target', y=col, ax=axes[idx], palette='Set2')
        axes[idx].set_title(col, fontsize=10)
    
    for idx in range(n_features, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'Boxplots by Target - {name.upper()}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'boxplots_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: boxplots_{name}.png')

## 8. Outlier Detection Report

In [ ]:
for name, df in datasets.items():
    print(f"\n--- Outlier Report: {name.upper()} ---")
    outlier_report = detect_outliers_iqr(df)
    display(outlier_report[outlier_report['Outlier Count'] > 0])

## 9. Pairplot (Heart Disease - Selected Features)

In [ ]:
# Pairplot for heart disease (selected features for readability)
if 'heart' in datasets:
    heart_df = datasets['heart']
    selected_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'target']
    existing = [c for c in selected_features if c in heart_df.columns]
    
    if len(existing) > 2:
        g = sns.pairplot(
            heart_df[existing], hue='target',
            palette='Set1', diag_kind='kde', height=2.5
        )
        g.fig.suptitle('Pairplot - Heart Disease (Selected Features)', y=1.02)
        plt.savefig(FIGURES_DIR / 'pairplot_heart.png', dpi=100, bbox_inches='tight')
        plt.show()
        print('Saved: pairplot_heart.png')

## 10. Statistical Summary

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"STATISTICAL SUMMARY: {name.upper()}")
    print(f"{'='*60}")
    print(f"\nSamples: {len(df)}")
    print(f"Features: {len(df.columns) - 1}")
    print(f"\nTarget distribution:")
    print(df['target'].value_counts())
    print(f"\nClass balance ratio: {df['target'].value_counts().min() / df['target'].value_counts().max():.3f}")
    print(f"\nDescriptive Statistics:")
    display(df.describe().round(3).T)

## Summary

### Key Findings:
- All datasets loaded successfully
- Target variables converted to binary classification
- Missing values identified and documented
- Outliers detected using IQR method
- All visualizations saved to `reports/figures/`